In [2]:
from pathlib import Path
import ast 
import pandas as pd

train_df = pd.read_csv(r"C:\Users\josie\OneDrive\Desktop\School\Capstone\Phishing Detection\MTPD CAPSTONE\data\raw\train_data.csv")
test_df = pd.read_csv(r"C:\Users\josie\OneDrive\Desktop\School\Capstone\Phishing Detection\MTPD CAPSTONE\data\raw\test_data.csv")

print("training conversations:", len(train_df))
print("testing conversations:", len(test_df))

training conversations: 450
testing conversations: 900


## Conversation identifiers

Each original conversation needs a permanent identifier before it is separated into message turns. The supplied training and testing origins are also retained for traceability. These source indicators are documentation fields and will not be used as model features.

In [3]:
train_working = train_df.copy()
test_working = test_df.copy()

train_working["source_split"] = "authors_train"
test_working["source_split"] = "authors_test"

train_working["source_row"] = range(len(train_working))
test_working["source_row"] = range(len(test_working))

train_working["conversation_id"] = [
    f"train_{row_number:04d}"
    for row_number in range(len(train_working))
]

test_working["conversation_id"] = [
    f"test_{row_number:04d}"
    for row_number in range(len(test_working))
]

working_df = pd.concat(
    [train_working, test_working], 
ignore_index=True
)

print("Combined conversations:", len(working_df)
    )

print(
    "Unique conversation IDs:", working_df["conversation_id"].nunique()
)

Combined conversations: 1350
Unique conversation IDs: 1350


In [4]:
print("Training working rows:", len(train_working))
print("Testing working rows:", len(test_working))
print("Combined working rows:", len(working_df))

print("\nFirst training IDs:")
print(train_working["conversation_id"].head().tolist())

print("\nFirst testing IDs:")
print(test_working["conversation_id"].head().tolist())

Training working rows: 450
Testing working rows: 900
Combined working rows: 1350

First training IDs:
['train_0000', 'train_0001', 'train_0002', 'train_0003', 'train_0004']

First testing IDs:
['test_0000', 'test_0001', 'test_0002', 'test_0003', 'test_0004']


## Parse Serialized Conversations
The original CSV stores each complete conversation as text representing a python list. The conversation field needs to be parsed into a list before individual entries can be normalized and separated into ordered turns. 

In [5]:
def parse_conversation(conversation_text): 
    """Convert a serialized conversation string into a python list."""

    conversation = ast.literal_eval(conversation_text)

    if not isinstance(conversation, list):
        raise ValueError("Conversation text did not parse into a list.")

    return conversation
working_df["parsed_conversation"] = working_df["conversation"].apply(parse_conversation)

print("Conversations parsed:", len(working_df)) 



Conversations parsed: 1350


In [6]:
working_df["number_of_entries"] = working_df["parsed_conversation"].apply(len)

print("Conversation entry counts:") 
display( working_df["number_of_entries"].value_counts().sort_index())

print("Non-list conversations:",
sum(not isinstance(conversation,list) 
    for conversation in working_df["parsed_conversation"]
                                   ))

Conversation entry counts:


number_of_entries
11    1350
Name: count, dtype: int64

Non-list conversations: 0


## Parsing Result 
 All 1350 serialized conversations were successfully organized into python lists. Every parsed conversation contains 11 ordered entries. The indivdiual entries still include a miscture od dictionaries, strogs, and nested list so they need to be normalized before turn level rows are created. 

In [7]:
import json
import re
import ast

In [8]:
# dictionaries and nested links 

def unwrap_entry(entry):
    """Unwrap entries that are already dictionaries or one-item lists."""
    if isinstance(entry, dict):
        return entry
    if isinstance(entry, list) and len(entry) == 1:
        return unwrap_entry(entry[0])
    
    return entry

In [9]:
#Test 

regular_entry = working_df.loc[
    0, "parsed_conversation"
][0]

unwrapped_entry = unwrap_entry(regular_entry)

print("Original type:", type(regular_entry).__name__)
print("Unwrapped type:", type(unwrapped_entry).__name__)
print("Fields:", unwrapped_entry.keys())

Original type: dict
Unwrapped type: dict
Fields: dict_keys(['Name', 'Message'])


In [10]:
#string entries parsing 

def parse_string_entry(entry):
    """Convert a serialized string entry into a dictionary."""
    text = entry.strip()

    for parser in (json.loads, ast.literal_eval):
        try:
            parsed_entry = parser(text)

            if isinstance(parsed_entry, dict):
                return parsed_entry
            if(
                isinstance(parsed_entry, list) 
                and len(parsed_entry) == 1
                and isinstance(parsed_entry[0], dict)
            ):
                return parsed_entry[0]
            
        except (ValueError, SyntaxError, json.JSONDecodeError):
            continue

    return None

In [11]:
# string test 

string_example = None
for conversation in working_df["parsed_conversation"]:
    for entry in conversation: 
        if isinstance(entry, str):
            string_example = entry
            break
    if string_example is not None:
        break

parsed_string_example = parse_string_entry(string_example)

print("Original type:", type(string_example).__name__) 
print("Parsed type:", type(parsed_string_example).__name__) 

if parsed_string_example is not None: 
    print("Fields:", parsed_string_example.keys())
            

Original type: str
Parsed type: dict
Fields: dict_keys(['Name', 'Message'])


In [12]:
# Normalize One Entry 

def normalize_entry(entry):
    """Convert an Entry into a standard Name and Message dictionary."""

    entry = unwrap_entry(entry)

    if isinstance(entry, str): 
        entry = parse_string_entry(entry)
    if not isinstance(entry, dict):
        return None

    if "Name" not in entry or "Message" not in entry:
        return None

    name = str(entry["Name"]).strip()
    message = str(entry["Message"]).strip()

    if not name or not message:
        return None

    return {
        "name": name,
        "message": message
    }

In [13]:
print(
  "regular entry:", 
  normalize_entry(regular_entry) is not None
 )

print(
 "string entry:",
 normalize_entry(string_example) is not None
)

regular entry: True
string entry: True


In [14]:
# Every Entry Test without Data Change 
successful_entries = 0
failed_entries = []

for _, row in working_df.iterrows():
    for turn_number, entry in enumerate(
        row["parsed_conversation"],
        start=1
    ):
        normalized = normalize_entry(entry)
        if normalized is None:
            failed_entries.append({
                "conversation_id": row["conversation_id"],
                "turn_number": turn_number,
                "entry_type": type(entry).__name__
                })
            
        else: 
            successful_entries += 1

print("Successfully Normalized Entries:", successful_entries)
print("Entries needing additional cleaning:", len(failed_entries))

            

Successfully Normalized Entries: 14662
Entries needing additional cleaning: 188


<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an i

## Initial Normalization

The intital Normalization process successfully standardized 14662 of the 14850 conversation entries. The 188 that were not standardized contain additional formatting irregularities and require further cleaning. All entries were accounted for with no record deletion. Repeated invalid escape warnings were from formatting in the original serialized conversation strings. 

In [15]:
#failure list table 

failed_entries_df = pd.DataFrame(failed_entries) 

print("Failed entries:", len(failed_entries_df)) 

print(
    "Conversations affected:", 
    failed_entries_df["conversation_id"].nunique()
)

display(failed_entries_df.head())

Failed entries: 188
Conversations affected: 77


,conversation_id,turn_number,entry_type
0,train_0057,6,str
1,train_0067,6,str
2,train_0084,8,str
3,train_0084,10,str
4,train_0118,5,str


In [16]:
#count failures by entry type
failure_type_counts = (
    failed_entries_df["entry_type"].value_counts()
    .value_counts()
    .rename_axis("entry type")
    .reset_index(name="failed entries")
)

display(failure_type_counts)

,entry type,failed entries
0,188,1


In [17]:
#count failures by source file 
failed_entries_df["source_split"] = (
    failed_entries_df["conversation_id"]
    .str.split("_")
    .str[0]
)

failed_source_counts = (
    failed_entries_df["source_split"].value_counts()
    .rename_axis("source file")
    .reset_index(name="failed entries")
)

display(failed_source_counts)

,source file,failed entries
0,test,141
1,train,47


In [18]:
#turns failure frequency 
failure_turn_counts = (
    failed_entries_df["turn_number"]
    .value_counts()
    .sort_index()
    .rename_axis("turn number")
    .reset_index(name="failed entries")
)

display(failure_turn_counts)

,turn number,failed entries
0,1,1
1,2,2
2,3,5
3,4,17
4,5,12
5,6,40
6,7,18
7,8,35
8,9,16
9,10,30


## Remaining Normalization Failures 

The initial parser accounted for 14850 entries but could not standardize 188 of them. Failures were analyzed by original entry type, source file, affected conversation and turn position. This will guid the next cleaning rule without deleting or manually altering the original data.

In [19]:
# additional cleaning analysis
failed_entries_df = pd.DataFrame(failed_entries)

print("Failed Entries:", len(failed_entries_df))

print(
    "conversations affected:",
    failed_entries_df["conversation_id"].nunique()
)

display(failed_entries_df.head())


Failed Entries: 188
conversations affected: 77


,conversation_id,turn_number,entry_type
0,train_0057,6,str
1,train_0067,6,str
2,train_0084,8,str
3,train_0084,10,str
4,train_0118,5,str


## Remaining Normalization Failures 

The initial normalization identified entries that could not be converted using the standard dictionary, nested-list, or serialized-string rules. Entries were retained and recorded by conversation id, turn number and original data type for further analysis. 

In [20]:
# failure entry type 
failure_type_counts = (
    failed_entries_df["entry_type"].value_counts()
   .rename_axis("entry_type")
   .reset_index(name="failed entries")
)

display(failure_type_counts)

,entry_type,failed entries
0,str,188


In [21]:
# failures from training or testing
failed_entries_df["source_file"] = (
    failed_entries_df["conversation_id"]
    .str.split("_")
    .str[0]
)

failure_source_counts = (
    failed_entries_df["source_file"].value_counts()
   .rename_axis("source_file")
   .reset_index(name="failed entries")
)

display(failure_source_counts)

,source_file,failed entries
0,test,141
1,train,47


### Failure Source 

The remaining normalization were grouped by their original source file. This determined whether the formatting problem is concetrated in the authors' training data, testing data, or distributed across both files 

### Failure position

The remaining failures were grouped by turn position to determine whether formatting problems occur throughout conversations or are concentrated at particular turns.

In [22]:
failure_turn_counts = (
    failed_entries_df["turn_number"]
    .value_counts()
    .sort_index()
    .rename_axis("turn_number")
    .reset_index(name="failed entries")
)

display(failure_turn_counts)

,turn_number,failed entries
0,1,1
1,2,2
2,3,5
3,4,17
4,5,12
5,6,40
6,7,18
7,8,35
8,9,16
9,10,30


### Failure summary

Most conversation entries were successfully normalized. The table below summarizes the remaining entries that require additional cleaning.

In [23]:
total_entries = 14850

failure_summary = pd.DataFrame({
    "Measure": [
        "Failed entries",
        "Affected Conversations",
        "Percentage of all entries"
    ],

    "Value" : [
        len(failed_entries_df),
        failed_entries_df["conversation_id"].nunique(),
        round(len(failed_entries_df) / total_entries * 100, 2)
    ]
})

display(failure_summary)

,Measure,Value
0,Failed entries,188.00
1,Affected Conversations,77.00
2,Percentage of all entries,1.27


## Inspect Unresolved Entries 
 The unresolved entries are retained with their original text so that formatting patterns can be identified before additional cleaning rules are developed. 

In [26]:
failed_entry_details =[]

for _, row in working_df.iterrows():
    for turn_number, entry in enumerate(
        row["parsed_conversation"],
        start=1
    ):
        if normalize_entry(entry) is None: 
            failed_entry_details.append({
                "conversation_id": row["conversation_id"],
                "source_split": row["source_split"],
                "turn_number": turn_number,
                "raw_entry": entry, 
            })

            failed_entry_details_df = pd.DataFrame(failed_entry_details)

            print("Unresolved entries:", len(failed_entry_details_df))

            display(failed_entry_details_df.head(10))

pd.set_option("display.max_colwidth", None)

display(
    failed_entry_details_df[
        [
            "conversation_id",
            "turn_number",
            "raw_entry"
        ]
    ].head(5)
)

Unresolved entries: 1


,conversation_id,source_split,turn_number,raw_entry
0,train_0057,authors_train,6,"{""Name"": ""Ruby"", ""Message"": ""Hello Marshall..."


Unresolved entries: 2


,conversation_id,source_split,turn_number,raw_entry
0,train_0057,authors_train,6,"{""Name"": ""Ruby"", ""Message"": ""Hello Marshall..."
1,train_0067,authors_train,6,"{\n ""Name"": ""Mamie"",\n ""Message"": ""Hi S..."


Unresolved entries: 3


,conversation_id,source_split,turn_number,raw_entry
0,train_0057,authors_train,6,"{""Name"": ""Ruby"", ""Message"": ""Hello Marshall..."
1,train_0067,authors_train,6,"{\n ""Name"": ""Mamie"",\n ""Message"": ""Hi S..."
2,train_0084,authors_train,8,"{""Name"": ""William"", ""Message"": ""Absolutely,..."


... (374 outputs left out)

,conversation_id,source_split,turn_number,raw_entry
0,train_0057,authors_train,6,"{""Name"": ""Ruby"", ""Message"": ""Hello Marshall..."
1,train_0067,authors_train,6,"{\n ""Name"": ""Mamie"",\n ""Message"": ""Hi S..."
2,train_0084,authors_train,8,"{""Name"": ""William"", ""Message"": ""Absolutely,..."
3,train_0084,authors_train,10,"{""Name"": ""William"", ""Message"": ""Of course..."
4,train_0118,authors_train,5,"{\n ""Name"": ""Maricela"",\n ""Message"": ""Hi ..."
5,train_0193,authors_train,4,"{""Name"": ""Carole"", ""Message"": ""Hello Tammie,..."
6,train_0193,authors_train,6,"{""Name"": ""Carole"", ""Message"": ""Hello Tammie, ..."
7,train_0193,authors_train,8,"{""Name"": ""Carole"", ""Message"": ""Hello Tammie, ..."
8,train_0193,authors_train,10,"{""Name"": ""Carole"", ""Message"": ""Hello Tammie,..."
9,train_0197,authors_train,9,"{\n ""Name"": ""Thomas"",\n ""Message"": ""De..."


Unresolved entries: 188


,conversation_id,source_split,turn_number,raw_entry
0,train_0057,authors_train,6,"{""Name"": ""Ruby"", ""Message"": ""Hello Marshall..."
1,train_0067,authors_train,6,"{\n ""Name"": ""Mamie"",\n ""Message"": ""Hi S..."
2,train_0084,authors_train,8,"{""Name"": ""William"", ""Message"": ""Absolutely,..."
3,train_0084,authors_train,10,"{""Name"": ""William"", ""Message"": ""Of course..."
4,train_0118,authors_train,5,"{\n ""Name"": ""Maricela"",\n ""Message"": ""Hi ..."
5,train_0193,authors_train,4,"{""Name"": ""Carole"", ""Message"": ""Hello Tammie,..."
6,train_0193,authors_train,6,"{""Name"": ""Carole"", ""Message"": ""Hello Tammie, ..."
7,train_0193,authors_train,8,"{""Name"": ""Carole"", ""Message"": ""Hello Tammie, ..."
8,train_0193,authors_train,10,"{""Name"": ""Carole"", ""Message"": ""Hello Tammie,..."
9,train_0197,authors_train,9,"{\n ""Name"": ""Thomas"",\n ""Message"": ""De..."


,conversation_id,turn_number,raw_entry
0,train_0057,6,"{""Name"": ""Ruby"", ""Message"": ""Hello Marshall, I'm glad to hear that you're keeping my perspective in mind as you gather your questions. If you're looking for additional insights or resources on the topic of AI and its impact on the future of work, I'd recommend checking out the following sources:\n- McKinsey Global Institute's report on 'Jobs Lost, Jobs Gained: What the Future of Work Will Mean for Jobs, Skills, and Wages'\n- World Economic Forum's report on 'The Future of Jobs'\n- Brookings Institution's report on 'The potential impact of artificial intelligence on the labor market'\nThese sources provide a comprehensive overview of the potential benefits and challenges of AI, as well as strategies for mitigating risks and maximizing opportunities. I look forward to our discussion and sharing more of my thoughts on this important topic.""}"
1,train_0067,6,"{\n ""Name"": ""Mamie"",\n ""Message"": ""Hi Sam,\n\nI'm glad to hear that you're interested in learning more about the role of education and awareness in promoting the use of solar energy. I believe that education and awareness are crucial factors in driving the adoption of solar energy. By educating individuals and businesses about the benefits of solar power, we can help them make informed decisions about their energy use and encourage them to switch to renewable sources.\n\nOne way to raise awareness about solar energy is through public campaigns and outreach programs. For example, the Solar Energy Industries Association (SEIA) in the US runs a program called ""Solar Ready Vets"" that provides training and job placement assistance to military veterans in the solar industry. This program not only helps veterans find employment, but also raises awareness about the benefits of solar energy and encourages more people to consider it as an option.\n\nAnother way to promote education and awareness is through partnerships between industry, government, and educational institutions. For example, the University of California, Berkeley, has partnered with several solar companies to develop a solar energy curriculum for high school students. This program provides students with hands-on experience in solar energy and helps them understand the science and technology behind it.\n\nI would be happy to discuss these examples in more detail and provide additional insights. Please let me know if there's anything else you'd like me to focus on.\n\nBest regards,\nMamie""\n}"
2,train_0084,8,"{""Name"": ""William"", ""Message"": ""Absolutely, Vickie. Ensuring the ethical use of AI and machine learning in mental health and wellness is a critical aspect of our work. Some of the guidelines and best practices we follow include:\n\n1. Obtaining informed consent from individuals before collecting and using their data.\n2. Ensuring data privacy and security by implementing robust encryption and anonymization techniques.\n3. Collaborating with mental health professionals to ensure that our interventions are evidence-based and clinically sound.\n4. Regularly reviewing and updating our algorithms and models to minimize the risk of bias and ensure fairness.\n5. Providing transparency and explainability in our AI systems, so that users can understand how decisions are made and have the opportunity to challenge them if necessary.""}"
3,train_0084,10,"{""Name"": ""William"", ""Message"": ""Of course, Vickie. While AI and machine learning have the potential to greatly benefit mental health and wellness, there are also several challenges and limitations to consider. Some of these include:\n\n1. The need for large and diverse datasets to train accurate and robust models.\n2. The difficulty of accurately measuring and quantifying mental health symptoms and outcomes.\n3. The potential for bias in AI systems, which can lead to unfair or inaccurate results.\n4. The need for ongoing validation and evaluation of AI-based interventions to ensure their effec

### Test Relaxed JSON Parsing

The unresolved entries appear to contain literal line breaks inside their Message fields. Relaxed JSON parsing is tested to determine whether these entries can be recovered without altering their content.

In [27]:
def parse_relaxed_json_entry(entry):
    """Parse JSON entries containing literal control characters."""

    if not isinstance(entry, str):
        return None

    try:
        parsed_entry = json.loads(
            entry.strip(),
            strict=False
        )
    except json.JSONDecodeError:
        return None

    if isinstance(parsed_entry, dict):
        return parsed_entry

    if (
        isinstance(parsed_entry, list)
        and len(parsed_entry) == 1
        and isinstance(parsed_entry[0], dict)
    ):
        return parsed_entry[0]

    return None

In [28]:
relaxed_json_successes = 0
relaxed_json_failures = []

for entry in failed_entry_details_df["raw_entry"]:
    parsed_entry = parse_relaxed_json_entry(entry)

    if parsed_entry is None:
        relaxed_json_failures.append(entry)
    else:
        relaxed_json_successes += 1

print(
    "Recovered with relaxed JSON:",
    relaxed_json_successes
)

print(
    "Still unresolved:",
    len(relaxed_json_failures)
)

Recovered with relaxed JSON: 107
Still unresolved: 81


In [29]:
for index, entry in enumerate(
    relaxed_json_failures[:5],
    start=1
):
    print(f"Unresolved example {index}:")
    print(repr(entry))
    print()

Unresolved example 1:
'     {\n  "Name": "Mamie",\n  "Message": "Hi Sam,\\n\\nI\'m glad to hear that you\'re interested in learning more about the role of education and awareness in promoting the use of solar energy. I believe that education and awareness are crucial factors in driving the adoption of solar energy. By educating individuals and businesses about the benefits of solar power, we can help them make informed decisions about their energy use and encourage them to switch to renewable sources.\\n\\nOne way to raise awareness about solar energy is through public campaigns and outreach programs. For example, the Solar Energy Industries Association (SEIA) in the US runs a program called "Solar Ready Vets" that provides training and job placement assistance to military veterans in the solar industry. This program not only helps veterans find employment, but also raises awareness about the benefits of solar energy and encourages more people to consider it as an option.\\n\\nAnother 

### Test Structural Field Extraction

Some remaining entries contain unescaped quotation marks or irregular backslashes inside otherwise consistent Name and Message structures. A structural parser is tested to extract the two required fields while preserving the message content.

In [30]:
def parse_structured_string_entry(entry):
    """Extract Name and Message from malformed JSON-like strings."""

    if not isinstance(entry, str):
        return None

    pattern = (
        r'^\s*\{\s*'
        r'"Name"\s*:\s*"(?P<name>.*?)"\s*,\s*'
        r'"Message"\s*:\s*"(?P<message>.*)"'
        r'\s*\}\s*$'
    )

    match = re.match(
        pattern,
        entry,
        flags=re.DOTALL
    )

    if match is None:
        return None

    name = match.group("name").strip()
    message = match.group("message").strip()

    # Remove stray backslashes before actual line breaks.
    message = re.sub(
        r"\\\r?\n",
        "\n",
        message
    )

    # Convert encoded line breaks into actual line breaks.
    message = (
        message
        .replace("\\n", "\n")
        .replace("\\r", "\r")
        .replace("\\t", "\t")
        .replace('\\"', '"')
    )

    if not name or not message:
        return None

    return {
        "Name": name,
        "Message": message
    }

In [31]:
structural_successes = 0
structural_failures = []

for entry in relaxed_json_failures:
    parsed_entry = parse_structured_string_entry(entry)

    if parsed_entry is None:
        structural_failures.append(entry)
    else:
        structural_successes += 1

print(
    "Recovered with structural parsing:",
    structural_successes
)

print(
    "Still unresolved:",
    len(structural_failures)
)

Recovered with structural parsing: 55
Still unresolved: 26


In [32]:
for index, entry in enumerate(
    structural_failures[:5],
    start=1
):
    print(f"Unresolved structural example {index}:")
    print(repr(entry))
    print()

Unresolved structural example 1:
'  {\n  "Name": "Sherry",\n  "Message": "Hi Mary, I\'m glad to hear that you\'re interested in learning more about the challenges I\'ve faced when implementing financial management strategies. I\'ve certainly encountered my fair share of obstacles, but I\'ve always found that the key to overcoming them is to approach each situation with a positive and proactive mindset. One example that comes to mind is when I worked with a client who was struggling to manage their cash flow. We implemented a budgeting and forecasting system that allowed them to better understand their financial situation and make more informed decisions. It wasn\'t easy, but by working together and staying focused on our goals, we were able to turn things around. I\'ll be happy to provide more examples if you\'d like. Let me know if there\'s anything else I can do to help."'

Unresolved structural example 2:
'  {\n  "Name": "Sherry",\n  "Message": "Hi Mary, I\'m glad to hear that you f

In [33]:
def parse_structured_string_entry(entry):
    """Extract Name and Message from malformed JSON-like strings."""

    if not isinstance(entry, str):
        return None

    patterns = [
        # Double-quoted format; final brace may be missing.
        (
            r'^\s*\{?\s*'
            r'"Name"\s*:\s*"(?P<name>.*?)"\s*,\s*'
            r'"Message"\s*:\s*"(?P<message>.*)"'
            r'\s*\}?\s*$'
        ),

        # Single-quoted format; final brace may be missing.
        (
            r"^\s*\{?\s*"
            r"'Name'\s*:\s*'(?P<name>.*?)'\s*,\s*"
            r"'Message'\s*:\s*'(?P<message>.*)'"
            r"\s*\}?\s*$"
        )
    ]

    match = None

    for pattern in patterns:
        match = re.match(
            pattern,
            entry,
            flags=re.DOTALL
        )

        if match is not None:
            break

    if match is None:
        return None

    name = match.group("name").strip()
    message = match.group("message").strip()

    # Remove stray backslashes before actual line breaks.
    message = re.sub(
        r"\\\r?\n",
        "\n",
        message
    )

    # Convert common encoded characters.
    message = (
        message
        .replace("\\n", "\n")
        .replace("\\r", "\r")
        .replace("\\t", "\t")
        .replace('\\"', '"')
        .replace("\\'", "'")
    )

    if not name or not message:
        return None

    return {
        "Name": name,
        "Message": message
    }

In [34]:
expanded_structural_successes = 0
final_structural_failures = []

for entry in structural_failures:
    parsed_entry = parse_structured_string_entry(entry)

    if parsed_entry is None:
        final_structural_failures.append(entry)
    else:
        expanded_structural_successes += 1

print(
    "Recovered with expanded structural parsing:",
    expanded_structural_successes
)

print(
    "Still unresolved:",
    len(final_structural_failures)
)

Recovered with expanded structural parsing: 23
Still unresolved: 3


In [35]:
for index, entry in enumerate(
    final_structural_failures,
    start=1
):
    print(f"Final unresolved example {index}:")
    print(repr(entry))
    print()

Final unresolved example 1:
"  [\n  {\n    'Name': 'James',\n    'Message': 'Of course, Ray! I'm glad to hear that you're open to considering this opportunity. The WHO is currently seeking individuals with your expertise to lead a groundbreaking research project on the impact of climate change on global health. This project has the potential to make a significant impact on the health of millions of people around the world, and your expertise in environmental science and public health would be invaluable to the team. I would be happy to provide you with more information about the project, including the specific responsibilities and expected outcomes, as well as the compensation and benefits package. Please let me know if you would like to discuss this further.'\n  }\n]"

Final unresolved example 2:
"      [\n  {\n    'Name': 'James',\n    'Message': 'Thank you for your response, Ray. I am available to discuss this opportunity further next week, and I would be happy to provide you with a

In [36]:
def parse_final_irregular_entry(entry):
    """Handle list wrappers and text added after a completed entry."""

    if not isinstance(entry, str):
        return None

    text = entry.strip()

    # Remove a one-item list wrapper.
    if text.startswith("[") and text.endswith("]"):
        text = text[1:-1].strip()

    # Try the existing structural parser first.
    parsed_entry = parse_structured_string_entry(text)

    if parsed_entry is not None:
        return parsed_entry

    # Remove commentary appearing after a completed dictionary.
    closing_patterns = [
        r'"\s*\}',
        r"'\s*\}"
    ]

    for pattern in closing_patterns:
        closing_match = re.search(
            pattern,
            text,
            flags=re.DOTALL
        )

        if closing_match is not None:
            dictionary_text = text[
                :closing_match.end()
            ]

            parsed_entry = parse_structured_string_entry(
                dictionary_text
            )

            if parsed_entry is not None:
                return parsed_entry

    return None

In [37]:
final_recovery_successes = 0
unrecoverable_entries = []

for entry in final_structural_failures:
    parsed_entry = parse_final_irregular_entry(entry)

    if parsed_entry is None:
        unrecoverable_entries.append(entry)
    else:
        final_recovery_successes += 1

print(
    "Recovered with final rule:",
    final_recovery_successes
)

print(
    "Unrecoverable entries:",
    len(unrecoverable_entries)
)

Recovered with final rule: 3
Unrecoverable entries: 0


## Final Entry parser 
The tested parsing rules are combined into one entry parser. The parser attempts relaxed JSON, structural field extraction, special irregular-entry handling, and Python literal parsing while preserving the original Name and Message content.

In [38]:
def parse_string_entry(entry):
    """Convert a serialized string entry into a dictionary."""

    if not isinstance(entry, str):
        return None

    text = entry.strip()

    # Method 1: relaxed JSON parsing
    try:
        parsed_entry = json.loads(
            text,
            strict=False
        )

        if isinstance(parsed_entry, dict):
            return parsed_entry

        if (
            isinstance(parsed_entry, list)
            and len(parsed_entry) == 1
            and isinstance(parsed_entry[0], dict)
        ):
            return parsed_entry[0]

    except json.JSONDecodeError:
        pass

    # Method 2: structural parsing
    parsed_entry = parse_structured_string_entry(text)

    if parsed_entry is not None:
        return parsed_entry

    # Method 3: final irregular formats
    parsed_entry = parse_final_irregular_entry(text)

    if parsed_entry is not None:
        return parsed_entry

    # Method 4: valid Python-literal formatting
    try:
        parsed_entry = ast.literal_eval(text)

        if isinstance(parsed_entry, dict):
            return parsed_entry

        if (
            isinstance(parsed_entry, list)
            and len(parsed_entry) == 1
            and isinstance(parsed_entry[0], dict)
        ):
            return parsed_entry[0]

    except (ValueError, SyntaxError):
        pass

    return None

In [39]:
successful_entries = 0
remaining_failures = []

for _, row in working_df.iterrows():
    for turn_number, entry in enumerate(
        row["parsed_conversation"],
        start=1
    ):
        normalized = normalize_entry(entry)

        if normalized is None:
            remaining_failures.append({
                "conversation_id": row["conversation_id"],
                "turn_number": turn_number,
                "entry_type": type(entry).__name__,
                "raw_entry": entry
            })
        else:
            successful_entries += 1

print(
    "Successfully normalized entries:",
    successful_entries
)

print(
    "Remaining failures:",
    len(remaining_failures)
)

Successfully normalized entries: 14850
Remaining failures: 0


## Create Normalized Conversations

Each parsed conversation is converted into an ordered list of standardized entries containing consistent name and message fields.

In [40]:
working_df["normalized_conversation"] = (
    working_df["parsed_conversation"].apply(
        lambda conversation: [
            normalize_entry(entry)
            for entry in conversation
        ]
    )
)

print(
    "Normalized conversations:",
    len(working_df)
)

print(
    "Entries in each conversation:"
)

display(
    working_df["normalized_conversation"]
    .apply(len)
    .value_counts()
    .sort_index()
)

Normalized conversations: 1350
Entries in each conversation:


normalized_conversation
11    1350
Name: count, dtype: int64

In [41]:
print("Working data columns:")

for column in working_df.columns:
    print("-", column)

Working data columns:
- conversation
- trait
- attack_success
- rationale
- scenario_class
- information_class
- attack_strategies
- label
- source_split
- source_row
- conversation_id
- parsed_conversation
- number_of_entries
- normalized_conversation


## Create Turn-Level Dataset

Each normalized conversation is expanded into one row per ordered turn. Only experiment-relevant metadata and traceability fields are retained. Supplied personality, outcome, rationale, and attack-strategy fields are excluded.

In [42]:
turn_df = (
    working_df[
        [
            "conversation_id",
            "source_split",
            "source_row",
            "scenario_class",
            "information_class",
            "label",
            "normalized_conversation"
        ]
    ]
    .explode(
        "normalized_conversation",
        ignore_index=True
    )
)

turn_df["turn_number"] = (
    turn_df.groupby("conversation_id")
    .cumcount()
    .add(1)
)

turn_df["speaker"] = (
    turn_df["normalized_conversation"]
    .apply(lambda entry: entry["name"])
)

turn_df["message"] = (
    turn_df["normalized_conversation"]
    .apply(lambda entry: entry["message"])
)

turn_df = turn_df.drop(
    columns="normalized_conversation"
)

turn_df = turn_df[
    [
        "conversation_id",
        "turn_number",
        "speaker",
        "message",
        "scenario_class",
        "information_class",
        "label",
        "source_split",
        "source_row"
    ]
]

print(
    "Turn-level rows:",
    len(turn_df)
)

print(
    "Unique conversations:",
    turn_df["conversation_id"].nunique()
)

display(turn_df.head())

Turn-level rows: 14850
Unique conversations: 1350


,conversation_id,turn_number,speaker,message,scenario_class,information_class,label,source_split,source_row
0,train_0000,1,Edward,"Dear Samuel,\n\nI hope this message finds you well. I am Edward, the Director of Research and Development at Innovatech, a leading technology company specializing in artificial intelligence and machine learning. I came across your impressive profile and extensive research in the field of AI ethics and was inspired by your dedication and passion.\n\nAt Innovatech, we are committed to fostering innovation and supporting the next generation of thought leaders in the AI industry. With this in mind, I would like to extend an offer for academic funding to support your ongoing research in AI ethics. This funding will provide you with the resources necessary to continue your groundbreaking work and make a significant impact in the field.\n\nPlease let me know if you are interested in discussing this opportunity further. I look forward to the possibility of working together to advance the boundaries of AI ethics.\n\nBest regards,\nEdward",AF,Personal Identifiable Information (PII),0.0,authors_train,0
1,train_0000,2,Samuel,"Dear Edward,\n\nThank you for your kind words and for reaching out to me. I am honored to learn that my work in AI ethics has caught your attention. I am grateful for your generous offer of academic funding to support my research.\n\nI would be delighted to discuss this opportunity further and learn more about Innovatech's commitment to fostering innovation and supporting thought leaders in the AI industry. Please let me know a convenient time for us to connect and explore this opportunity.\n\nThank you once again for your support and encouragement.\n\nBest regards,\nSamuel",AF,Personal Identifiable Information (PII),0.0,authors_train,0
2,train_0000,3,Edward,"Dear Samuel,\n\nI'm thrilled to hear of your enthusiasm for this opportunity. Your work in AI ethics is truly impressive and aligns perfectly with our mission at Innovatech. I'm confident that our collaboration will yield significant advancements in the field.\n\nHow does next Tuesday at 2 PM EST sound for our initial discussion? We can use LinkedIn's audio call feature for our conversation. If this time doesn't work for you, please let me know your availability, and I'll do my best to accommodate it.\n\nLooking forward to our discussion and the opportunity to support your research.\n\nBest regards,\nEdward",AF,Personal Identifiable Information (PII),0.0,authors_train,0
3,train_0000,4,Samuel,"Dear Edward,\n\nThank you for your prompt response and the proposed time for our initial discussion. I am available next Tuesday at 2 PM EST and look forward to our conversation via LinkedIn's audio call feature.\n\nI appreciate your flexibility and support in accommodating my schedule. I am excited about the potential collaboration between us and the opportunity to advance the field of AI ethics.\n\nThank you once again for your support and encouragement. I look forward to speaking with you next Tuesday.\n\nBest regards,\nSamuel",AF,Personal Identifiable Information (PII),0.0,authors_train,0
4,train_0000,5,Edward,"Dear Samuel,\n\nI'm delighted to hear that our proposed time works for you. I am equally excited about our collaboration and the opportunity to advance the field of AI ethics. Your work has the potential to make a significant impact, and I am honored to support it.\n\nI look forward to our conversation next Tuesday at 2 PM EST. In the meantime, if you have any questions or need further information, please do not hesitate to contact me.\n\nBest regards,\nEdward",AF,Personal Identifiable Information (PII),0.0,authors_train,0


## Construct Cumulative Conversation History

A cumulative-history representation is created for each turn. Each record contains the current message and all preceding messages from the same conversation while excluding future turns.


In [43]:
turn_df = turn_df.sort_values(
    [
        "conversation_id",
        "turn_number"
    ]
).reset_index(drop=True)

turn_df["turn_text"] = (
    turn_df["speaker"]
    + ": "
    + turn_df["message"]
)

turn_df["cumulative_history"] = (
    turn_df.groupby(
        "conversation_id"
    )["turn_text"]
    .transform(
        lambda messages: [
            "\n\n".join(
                messages.iloc[:index + 1]
            )
            for index in range(len(messages))
        ]
    )
)

print(
    "Rows with cumulative history:",
    turn_df["cumulative_history"].notna().sum()
)

display(
    turn_df[
        [
            "conversation_id",
            "turn_number",
            "turn_text",
            "cumulative_history"
        ]
    ].head(3)
)

Rows with cumulative history: 14850


,conversation_id,turn_number,turn_text,cumulative_history
0,test_0000,1,"Eric: Hello Frank, I hope this message finds you well. I am reaching out to you from the prestigious research organization, The Institute for Advanced Studies in Sustainable Technologies (IASST). Our organization is dedicated to supporting cutting-edge research in sustainable technologies, and we are currently seeking to fund promising academic projects in this field.\n\nI came across your impressive profile and extensive background in renewable energy systems, particularly your work on solar panel efficiency and waste-to-energy conversion. Given your expertise and our shared interest in sustainable technologies, I believe that our organization could provide valuable support for your research endeavors.\n\nPlease feel free to share any ongoing or proposed projects related to sustainable technologies, and we will be more than happy to discuss potential funding opportunities. I look forward to hearing from you and learning more about your work.\n\nBest regards,\nEric","Eric: Hello Frank, I hope this message finds you well. I am reaching out to you from the prestigious research organization, The Institute for Advanced Studies in Sustainable Technologies (IASST). Our organization is dedicated to supporting cutting-edge research in sustainable technologies, and we are currently seeking to fund promising academic projects in this field.\n\nI came across your impressive profile and extensive background in renewable energy systems, particularly your work on solar panel efficiency and waste-to-energy conversion. Given your expertise and our shared interest in sustainable technologies, I believe that our organization could provide valuable support for your research endeavors.\n\nPlease feel free to share any ongoing or proposed projects related to sustainable technologies, and we will be more than happy to discuss potential funding opportunities. I look forward to hearing from you and learning more about your work.\n\nBest regards,\nEric"
1,test_0000,2,"Frank: Hello Eric, I'm thrilled to receive your message and learn about the incredible work being done at the Institute for Advanced Studies in Sustainable Technologies (IASST). I share your passion for sustainable technologies and am excited about the potential to collaborate.\n\nCurrently, I'm working on two projects that might be of interest to you. The first is an innovative approach to improving solar panel efficiency by using advanced materials and nanotechnology. The second project involves developing a more efficient waste-to-energy conversion process, focusing on maximizing energy output and minimizing environmental impact.\n\nI would be delighted to discuss these projects further and explore potential funding opportunities with your esteemed organization. Please let me know how you would like to proceed, and I look forward to our collaboration.\n\nBest regards,\nFrank","Eric: Hello Frank, I hope this message finds you well. I am reaching out to you from the prestigious research organization, The Institute for Advanced Studies in Sustainable Technologies (IASST). Our organization is dedicated to supporting cutting-edge research in sustainable technologies, and we are currently seeking to fund promising academic projects in this field.\n\nI came across your impressive profile and extensive background in renewable energy systems, particularly your work on solar panel efficiency and waste-to-energy conversion. Given your expertise and our shared interest in sustainable technologies, I believe that our organization could provide valuable support for your research endeavors.\n\nPlease feel free to share any ongoing or proposed projects related to sustainable technologies, and we will be more than happy to discuss potential funding opportunities. I look forward to hearing from you and learning more about your work.\n\nBest regards,\nEric\n\nFrank: Hello Eric, I'm thrilled to receive your message and learn about t

## Validate Turn-Level Data

The turn-level dataset is checked for missing values, correct turn numbering, and cumulative histories that contain only the current and preceding turns.

In [45]:
required_columns = [
    "conversation_id",
    "turn_number",
    "speaker",
    "message",
    "scenario_class",
    "information_class",
    "label",
    "cumulative_history"
]

print("Missing required values:")

display(
    turn_df[required_columns]
    .isna()
    .sum()
)

turn_count_check = (
    turn_df.groupby("conversation_id")
    ["turn_number"]
    .apply(list)
    .apply(
        lambda turns: turns == list(range(1, 12))
    )
)

print(
    "Conversations with correct turn order:",
    turn_count_check.sum()
)

history_errors = []

for conversation_id, group in turn_df.groupby(
    "conversation_id"
):
    expected_turns = []

    for _, row in group.sort_values(
        "turn_number"
    ).iterrows():
        expected_turns.append(row["turn_text"])

        expected_history = "\n\n".join(
            expected_turns
        )

        if row["cumulative_history"] != expected_history:
            history_errors.append({
                "conversation_id": conversation_id,
                "turn_number": row["turn_number"]
            })

print(
    "History construction errors:",
    len(history_errors)
)

Missing required values:


conversation_id       0
turn_number           0
speaker               0
message               0
scenario_class        0
information_class     0
label                 0
cumulative_history    0
dtype: int64

Conversations with correct turn order: 1350
History construction errors: 0


## Save Processed Turn-Level Data

The validated turn-level dataset is saved separately from the original source files. Raw SE-VSim data remains unchanged.

In [46]:
def find_project_root(start_path):
    """Find the project folder containing data and notebooks."""

    candidates = [
        start_path,
        *start_path.parents
    ]

    for candidate in candidates:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root."
    )


project_root = find_project_root(
    Path.cwd()
)

processed_data_directory = (
    project_root
    / "data"
    / "processed"
)

processed_data_directory.mkdir(
    parents=True,
    exist_ok=True
)

processed_data_path = (
    processed_data_directory
    / "turn_level_data.csv"
)

turn_df.to_csv(
    processed_data_path,
    index=False
)

print(
    "Saved processed data to:",
    processed_data_path
)

print(
    "Saved rows:",
    len(turn_df)
)

Saved processed data to: c:\Users\josie\OneDrive\Desktop\School\Capstone\Phishing Detection\MTPD CAPSTONE\data\processed\turn_level_data.csv
Saved rows: 14850


In [47]:
saved_turn_df = pd.read_csv(
    processed_data_path
)

print(
    "Reloaded rows:",
    len(saved_turn_df)
)

print(
    "Reloaded conversations:",
    saved_turn_df["conversation_id"].nunique()
)

Reloaded rows: 14850
Reloaded conversations: 1350
